In [ ]:
import time
import requests
import pandas as pd
from datetime import date

API = "https://api.jolpi.ca/ergast/f1"          # successor to Ergast, free, no API key
SEASONS = range(2022, date.today().year + 1)    # from 2022 (ground effect car era)

def fetch_results(season):
    rows, offset = [], 0
    while True:
        r = requests.get(f"{API}/{season}/results.json", params={"limit": 100, "offset": offset}).json()
        for race in r["MRData"]["RaceTable"]["Races"]:
            for h in race["Results"]:
                rows.append({
                    "season": int(race["season"]),
                    "round": int(race["round"]),
                    "race": race["raceName"],
                    "circuit": race["Circuit"]["circuitId"],
                    "driver": h["Driver"]["familyName"],
                    "team": h["Constructor"]["name"],
                    "grid": int(h["grid"]),
                    "position": int(h["position"]),
                    "points": float(h["points"]),
                })
        offset += 100
        if offset >= int(r["MRData"]["total"]):
            break
        time.sleep(0.5)   # the API is rate limited, don't spam it
    return rows

df_raw = pd.DataFrame([row for s in SEASONS for row in fetch_results(s)])
df_raw["grid"] = df_raw["grid"].replace(0, 22)   # grid 0 = pit lane start → treat as last

print(df_raw.shape)
print(df_raw[df_raw["position"] == 1]["driver"].value_counts())   # who wins the most?
print(df_raw.tail())

In [ ]:
def make_features(df):
    df = df.sort_values(["season", "round"]).copy()
    per_driver = df.groupby("driver")

    # shift(1) → only use PREVIOUS races, so the model can't peek at the result of the race itself
    # 1. average finish & grid position over the last 5 races
    df["form_position"] = per_driver["position"].transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean())
    df["form_grid"] = per_driver["grid"].transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean())

    # 2. points & wins so far this season
    per_season = df.groupby(["season", "driver"])
    df["season_points"] = per_season["points"].transform(lambda s: s.shift(1).cumsum()).fillna(0)
    df["season_wins"] = per_season["position"].transform(lambda s: (s == 1).astype(float).shift(1).cumsum()).fillna(0)

    # 3. how fast the car is: average position of the team's 2 drivers over the last 5 races
    team = df.groupby(["season", "round", "team"], as_index=False)["position"].mean()
    team["team_form"] = team.groupby("team")["position"].transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean())
    df = df.merge(team[["season", "round", "team", "team_form"]], on=["season", "round", "team"])

    # 4. driver's average position at this circuit in previous years
    df["form_circuit"] = df.groupby(["driver", "circuit"])["position"].transform(lambda s: s.shift(1).expanding().mean())
    df["form_circuit"] = df["form_circuit"].fillna(df["form_position"])   # never raced here → use regular form
    return df

df = make_features(df_raw)
df["won"] = (df["position"] == 1).astype(int)
df = df.dropna()   # each driver's first race has no history yet → just drop it

FEATURES = ["form_position", "form_grid", "team_form", "form_circuit", "season_points", "season_wins"]
print(df.shape)
print(df[["season", "race", "driver"] + FEATURES + ["won"]].tail(10))

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report

device = "cuda" if torch.cuda.is_available() else "cpu"
CLASSES = ["Lost", "Won"]

# no shuffling: we want to predict the future from the past → last 10 races are the test data
race_key = df["season"] * 100 + df["round"]
test_races = sorted(race_key.unique())[-10:]
test = race_key.isin(test_races)

X_train, y_train = df.loc[~test, FEATURES].values, df.loc[~test, "won"].values
X_test, y_test = df.loc[test, FEATURES].values, df.loc[test, "won"].values

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

def to_tensor(X, y):
    return TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long))

train_loader = DataLoader(to_tensor(X_train, y_train), batch_size=32, shuffle=True)

model = nn.Sequential(
    nn.Linear(len(FEATURES), 32), nn.ReLU(),
    nn.Linear(32, 16), nn.ReLU(),
    nn.Linear(16, 2),
).to(device)

# only 1 winner out of ~20 drivers → give the "Won" class more weight so the model doesn't ignore it
loss_fn = nn.CrossEntropyLoss(weight=torch.tensor([1.0, 19.0]).to(device))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
EPOCHS = 50

# training
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        loss = loss_fn(model(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"epoch {epoch+1} | loss {total_loss / len(train_loader):.4f}")

In [ ]:
# evaluation
model.eval()
with torch.no_grad():
    prob = torch.softmax(model(torch.tensor(X_test, dtype=torch.float32).to(device)), dim=1)[:, 1].cpu().numpy()

print(confusion_matrix(y_test, prob > 0.5))
print(classification_report(y_test, prob > 0.5, target_names=CLASSES, zero_division=0))

# more important: in each race, did the driver with the highest chance actually win?
results = df.loc[test, ["season", "race", "driver", "position"]].copy()
results["prob"] = prob

correct = 0
for (season, race), r in results.groupby(["season", "race"], sort=False):
    pick = r.loc[r["prob"].idxmax(), "driver"]
    actual = r.sort_values("position").iloc[0]["driver"]
    correct += pick == actual
    print(f"{season} {race:28} | predicted {pick:12} | actual {actual}")

print(f"\ncorrect predictions: {correct} of {len(test_races)} races")

In [ ]:
# find the next GP from this season's schedule
schedule = requests.get(f"{API}/current.json").json()["MRData"]["RaceTable"]["Races"]
upcoming = [r for r in schedule if r["date"] >= str(date.today())]
assert upcoming, "this season is over, no upcoming GP yet"
gp = upcoming[0]

# driver line-up taken from the last race
last = df_raw[df_raw["season"] == df_raw["season"].max()]
last = last[last["round"] == last["round"].max()]

new = last[["driver", "team"]].copy()
new["season"] = int(gp["season"])
new["round"] = int(gp["round"])
new["race"] = gp["raceName"]
new["circuit"] = gp["Circuit"]["circuitId"]
# grid, position, points stay empty (the race hasn't happened yet), features still come from previous races

new_features = make_features(pd.concat([df_raw, new]))
new_features = new_features[(new_features["season"] == new["season"].iloc[0]) & (new_features["round"] == new["round"].iloc[0])]
new_features = new_features.dropna(subset=FEATURES).copy()   # rookie drivers have no history yet

with torch.no_grad():
    X_new = torch.tensor(scaler.transform(new_features[FEATURES].values), dtype=torch.float32).to(device)
    prob = torch.softmax(model(X_new), dim=1)[:, 1].cpu()

new_features["chance"] = (prob / prob.sum()).numpy()   # only 1 can win → make the chances add up to 100%

print(f"{gp['raceName']} ({gp['date']})\n")
for _, r in new_features.sort_values("chance", ascending=False).head(10).iterrows():
    print(f"{r['driver']:12} {r['team']:15} {r['chance']:.0%}")